In [1]:
uv add docling

/home/mikealson/anaconda3/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [2]:
from docling.document_converter import DocumentConverter

source = "/home/mikealson/Gen_AI_Notes/GenAiNote.txt"

converter = DocumentConverter()
result = converter.convert(source)

# exporting to extracted_Markdown as markdown.
Extracted_Markdown=result.document.export_to_markdown()

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


In [3]:
Extracted_Markdown[:1000]

'================================================================================ GEN AI STUDY NOTES\n\n# Daily update: 04 Jul 2026\n\n================================================================================\n\n================================================================================\n\n1. TEMPERATURE ================================================================================\n\nTemperature (high and low): controls the creativity / randomness of the model.\n\n- High temperature  (0.8 - 1.0) -&gt; more creative / random outputs\n- Low temperature   (0.0 - 0.3) -&gt; more deterministic / focused outputs\n- Medium temperature (0.4 - 0.7) -&gt; balanced between creativity and focus\n\nOther sampling parameters:\n\n- Top-p (nucleus sampling): considers only tokens whose cumulative probability reaches p (e.g. top\\_p=0.9 means only the top 90% likely tokens are considered)\n- Top-k: limits the model to the k most likely next tokens\n- Top-p and temperature are often used 

In [4]:
uv add langchain langchain-text-splitters bs4 requests

/home/mikealson/anaconda3/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [5]:
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()
#DUMMY_KEY_REDACTED

 ········


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Configure the splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,      # Maximum characters per chunk
    chunk_overlap=150,    # Overlap between consecutive chunks
    separators=["\n\n", "\n", " ", ""] # Split order priority
)

chunks = splitter.split_text(Extracted_Markdown)

# for i, chunk in enumerate(chunks):
#     print(f"Chunk {i+1}: {repr(chunk)}")
print(len(chunks))


139


In [11]:
uv add langchain-huggingface

/home/mikealson/anaconda3/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
from dotenv import load_dotenv

load_dotenv() # loads the .env file

secret_key = os.getenv("HUGGINGFACE_API_KEY")
print(secret_key)


hf_DUMMY_TOKEN_REDACTED


In [13]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="scaleway",
    api_key=secret_key,
)

result = client.feature_extraction(
    chunks,
    model="Qwen/Qwen3-Embedding-8B",
)

In [14]:
len(result)
result

array([[ 1.6694900e-02,  1.9163864e-02, -3.6005725e-03, ...,
        -4.0873845e-05, -7.6420316e-03, -2.5042349e-02],
       [-3.9349911e-03,  7.1995761e-03,  1.1367752e-03, ...,
        -2.4921610e-03, -1.5667095e-03, -2.5067352e-02],
       [-8.8650137e-03, -2.5059597e-03,  1.2584118e-04, ...,
        -1.4948846e-02, -2.8101513e-03, -1.2051783e-02],
       ...,
       [ 4.9223774e-03, -6.0788393e-03, -6.7905085e-03, ...,
        -1.4351991e-02, -5.7526575e-03,  7.5911358e-03],
       [ 1.2844698e-02, -1.0311110e-02, -2.3686092e-02, ...,
        -1.3669586e-02, -9.4862217e-03,  2.4275299e-02],
       [ 2.2096060e-02, -1.4750613e-02, -7.1662897e-03, ...,
        -1.1884097e-02, -5.9121894e-03,  1.2421569e-02]],
      shape=(139, 4096), dtype=float32)

In [15]:
uv add chromadb numpy

/home/mikealson/anaconda3/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [16]:
from langchain_chroma import Chroma
import numpy as np

vector_store = Chroma(
    collection_name="high_dim_collection",
    persist_directory="./hybrid_search_db" 
)

In [17]:
import numpy as np

# Convert the 2D array of embeddings directly into a standard Python list of lists
# This keeps each chunk's 4096-dimensional vector distinct instead of merging them.
embeddings_list = np.array(result).tolist()

# Save your entire dataset into your local directory
vector_store.add_texts(
    texts=chunks,
    embeddings=embeddings_list, # Passed correctly as a list of 139 vectors
    metadatas=[{"source": "study_notes"}] * len(chunks),
    ids=[f"doc_{i}" for i in range(len(chunks))]
)

print(f"Successfully stored {len(chunks)} chunks and 4096-dim vectors in './hybrid_search_db'!")

Successfully stored 139 chunks and 4096-dim vectors in './hybrid_search_db'!


#phase 2

In [25]:
from langchain_community.retrievers import BM25Retriever

# 1. Initialize your two retrievers safely
semantic_retriever = vector_store.as_retriever(search_kwargs={"k": 3})
keyword_retriever = BM25Retriever.from_texts(chunks)
keyword_retriever.k = 3

# 2. Manual Reciprocal Rank Fusion (RRF) Engine
def custom_hybrid_search(query, semantic_weight=0.5, keyword_weight=0.5, c=60):
    # Fetch results from both pipelines
    semantic_docs = semantic_retriever.invoke(query)
    keyword_docs = keyword_retriever.invoke(query)
    
    rrf_scores = {}
    doc_map = {}
    
    # Score semantic results
    for rank, doc in enumerate(semantic_docs):
        content = doc.page_content
        doc_map[content] = doc
        rrf_scores[content] = rrf_scores.get(content, 0) + semantic_weight * (1.0 / (rank + 1 + c))
        
    # Score keyword results
    for rank, doc in enumerate(keyword_docs):
        content = doc.page_content
        doc_map[content] = doc
        rrf_scores[content] = rrf_scores.get(content, 0) + keyword_weight * (1.0 / (rank + 1 + c))
        
    # Sort documents by their combined RRF score
    sorted_contents = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return [doc_map[content] for content in sorted_contents]

print("Custom Hybrid Search Engine compiled successfully without relying on core imports!")

Custom Hybrid Search Engine compiled successfully without relying on core imports!


In [27]:
query = "MCP"

# Invoke our custom math-backed pipeline
retrieved_docs = custom_hybrid_search(query)

print(f"Total documents retrieved: {len(retrieved_docs)}\n")
for i, doc in enumerate(retrieved_docs[:3]):  # Show top 3 distinct matches
    print(f"--- Rank {i+1} ---")
    print(doc.page_content)

Total documents retrieved: 4

--- Rank 1 ---
MCP (Model Context Protocol) is an open standard that lets LLMs interact with EXTERNAL TOOLS AND DATA SOURCES in a structured way.

Think of it as "USB-C for AI" -- a universal connector that lets any LLM work with any tool or data source without custom integrations.

--- Architecture ---

Host/Client (OpenWork, Claude) &lt;--&gt; MCP Server (tools, resources, prompts)

--- Three Core Decorators ---

1. @mcp.tool -- exposes a function as an ACTION the LLM can call Use for: API calls, calculations, file ops, DB queries
2. @mcp.resource -- exposes READ-ONLY DATA via a URI Use for: files, config, DB records
3. @mcp.prompt -- defines a reusable PROMPT TEMPLATE Use for: standard workflows, guided interactions

--- Key Benefits ---
--- Rank 2 ---
--- Key Benefits ---

- Standardization: one protocol for all tool integrations
- Security: host controls which MCP servers are allowed
- Reusability: share MCP servers across different LLM apps
- Simplic

In [32]:
# 1. Define the user's natural language question
query = "What is mcp"

# 2. Extract relevant documents using your custom RRF pipeline
retrieved_docs = custom_hybrid_search(query)

# 3. Join the contents of the retrieved text chunks together with clear dividers
context_text = "\n\n---\n\n".join([doc.page_content for doc in retrieved_docs[:3]])

# 4. Construct a clear, constrained system prompt instructing the model to rely on your internal docs
prompt_template = f"""You are a helpful engineering study assistant. Use the internal notes context provided below to answer the user's question accurately. 
If the answer cannot be found in the context, state that you don't know based on the provided notes.

Internal Notes Context:
{context_text}

Question: {query}
Answer:"""

print("RAG prompt successfully constructed with real-time internal context vectors!")
print(prompt_template)

RAG prompt successfully constructed with real-time internal context vectors!
You are a helpful engineering study assistant. Use the internal notes context provided below to answer the user's question accurately. 
If the answer cannot be found in the context, state that you don't know based on the provided notes.

Internal Notes Context:
--- Key Benefits ---

- Standardization: one protocol for all tool integrations
- Security: host controls which MCP servers are allowed
- Reusability: share MCP servers across different LLM apps
- Simplicity: write a Python function, add a decorator, done

--- MCP Servers (Examples) ---

Pre-built: Filesystem, GitHub, Database, Web, Slack, Puppeteer Custom: pip install mcp &amp;&amp; decorate your functions

# 


32. AGENTIC AI &amp; RAG COMPONENTS

# ponytail: condensed overview -- see Sections 20, 21 for full detail

--- What is an AI Agent? ---

Your note: "what is ai agent? an ai agent is a software program powered by an llm tha can percieve-&gt;rea

In [33]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=secret_key,
)

completion = client.chat.completions.create(
    model="Qwen/Qwen3.6-27B:featherless-ai",
    messages=[
        {
            "role": "user",
            "content": prompt_template   
        }
    ],
)

response=completion.choices[0].message


In [40]:
#response.content

In [39]:
# 1. Properly extract the text string from the ChatCompletionMessage object
raw_content = completion.choices[0].message.content

# 2. Strip out the internal <think> chain-of-thought blocks if present
if "</think>" in raw_content:
    clean_response = raw_content.split("</think>")[-1].strip()
else:
    clean_response = raw_content.strip()

# 3. Print the clean result
print("=== RAG System Answer ===")
print(clean_response)

=== RAG System Answer ===
Based on the provided notes, **MCP** stands for **Model Context Protocol**. It is an open standard that enables LLMs to interact with external tools and data sources in a structured way. The notes describe it as the **"USB-C for AI"**—a universal connector that allows any LLM to work with any tool or data source without needing custom integrations.

Key details from your notes:
- **Architecture:** Operates on a `Host/Client` (e.g., Claude, OpenWork) ↔ `MCP Server` model, where the server provides tools, resources, and prompts.
- **Three Core Decorators:**
  - `@mcp.tool`: Exposes a function as an **action** the LLM can call (e.g., API calls, calculations, DB queries).
  - `@mcp.resource`: Exposes **read-only data** via a URI (e.g., files, config, DB records).
  - `@mcp.prompt`: Defines a reusable **prompt template** (e.g., standard workflows, guided interactions).
- **Key Benefits:** 
  - **Standardization:** One protocol for all tool integrations
  - **Securi